# Debugging Errors with ErrorCategorizer() (refer to PROGRESS_NOTES/v03.md)

**Issue with _check_entity_detection():**

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path("/Users/robertagarcia/Desktop/learning/bert_symptom_ner")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from error_analysis.error_categorization import ErrorCategorizer, ErrorTaxonomy
from error_analysis.error_taxonomy import ErrorTaxonomy, BehaviouralExample

error_cat = ErrorCategorizer()
print(ErrorTaxonomy())

# Example that will lead to malfunction of _check_boundaries

In [ ]:
%pdb on
example = "No fever reported."
example2 = BehaviouralExample(
    example=example,
    entities_with_labels=[
        {"ent": "fever", "start": 3, "end": 8, "label": "SYMPTOM_NEG"}  # gold "fever" at 3–8
    ]
)
expected_start, expected_end = example2.entities_with_labels[0]["start"], example2.entities_with_labels[0]["end"]
print(f"Ground truth span: {example2.example[expected_start:expected_end]}")

predicted_spans2_list = [
    # span text contains "fever" but sits mostly BEFORE the gold position
    # overlap: max(3,0)=3 to min(8,4)=4 → 1 char. 1/5 = 20% < 50%
    {"text": "No fever", "start": 1, "end": 5, "label": "SYMPTOM_POS"}
]

predicted_spans2 = predicted_spans2_list[0]
predicted_start, predicted_end = predicted_spans2["start"], predicted_spans2["end"]
print(f"predicted span: {predicted_spans2["text"][predicted_start:predicted_end]}")


# Calculate overlap
overlap_start = max(expected_start, predicted_start)
overlap_end = min(expected_end, predicted_end)
print(f"overlap start: {overlap_start} and overlap end: {overlap_end}")
overlap_length = max(0, overlap_end - overlap_start)
print("overlap length: ", overlap_length)   
expected_length = expected_end - expected_start
predicted_length = predicted_end - predicted_start
print(f"expected length: {expected_length}, predicted length: {predicted_length}")
# based on the logic of the function, will it overlap > 50%?
if not(overlap_length  > expected_length * 0.5):
    print("This will trigger an error in the fuction _check_boundaries")

result2 = error_cat._check_entity_detection(example=example2, spans=predicted_spans2_list)
print("RESULTS: ", result2)
print("\nError counts:", dict(error_cat.error_counts))
# → present_errors: [], missing_entities: [], false_positives: []
# → error_counts: {}   ← SILENT: no errors recorded at all

# Testing a new _is_entity_in_spans
_is_entity_in_spans should require character-position overlap, not text substring containment. Otherwise "fever" matches "No fever reported yesterday" even when the spans barely touch 

In [ ]:
def _is_entity_in_spans(self, entity, spans):
    """Match a gold entity to the predicted span with the largest character overlap.
    Returns the span index, or None if no span overlaps the gold position at all.
    Prints all steps for debugging.
    """
    e_start, e_end = entity["start"], entity["end"]
    print(f"Gold entity: '{entity['ent']}' [{e_start}:{e_end}], label: {entity['label']}")
    best_idx, best_overlap = None, 0
    for idx, span in enumerate(spans):
        print(f"\nChecking predicted span {idx}: '{span['text']}' "
              f"[{span['start']}:{span['end']}], label: {span['label']}")
        if span["label"] == "O":
            print("  Skipping span with 'O' label.")
            continue
        # Calculate character overlap
        overlap_start = max(e_start, span["start"])
        overlap_end = min(e_end, span["end"])
        overlap = max(0, overlap_end - overlap_start)
        print(f"  Overlap start: {overlap_start}, overlap end: {overlap_end}")
        print(f"  Calculated overlap: {overlap} "
              f"(current best: {best_overlap} at idx {best_idx})")
        if overlap > best_overlap:
            best_idx, best_overlap = idx, overlap
            print(f"  New best: idx {best_idx}, overlap {best_overlap}")
    print(f"\nBest match index: {best_idx} (overlap: {best_overlap})")
    return best_idx

In [ ]:
text = "Sneezing  is very common during the winter but the patient denies sneezing."
#       0         1         2         3         4         5         6         7
#       0123456789012345678901234567890123456789012345678901234567890123456789012345

# Gold: only the SECOND "sneezing" is a real (negative) symptom
gold = {"ent": "sneezing", "start": 66, "end": 74, "label": "SYMPTOM_NEG"}

spans = [
    {"text": "Sneezing", "start": 0,  "end": 8,  "label": "SYMPTOM_POS"},   # wrong location
    {"text": "sneezing", "start": 66, "end": 74, "label": "SYMPTOM_NEG"},   # right
]
# span 0 overlap = 0
# span 1 overlap = 8  → best
# returns 1

results = _is_entity_in_spans(
    entity=gold,
    spans=spans
)